In [1]:
import sys
import os

# SPARK_HOME path
os.environ["SPARK_HOME"] = "/home/fragkiska/spark"

# Add pyspark to Python path
sys.path.append("/home/fragkiska/spark/python")
sys.path.append("/home/fragkiska/spark/python/lib/py4j-0.10.9.7-src.zip")


import pyspark
from pyspark.sql import SparkSession

In [ ]:
from pyspark.sql import SparkSession
from sedona.spark import *

existing_spark = SparkSession.getActiveSession()
if existing_spark:
    existing_spark.stop()

spark = SparkSession.builder \
    .appName("Query4") \
    .config("spark.jars.packages", 
            "org.apache.sedona:sedona-spark-shaded-3.4_2.12:1.6.1,"
            "org.datasyslab:geotools-wrapper:1.6.1-28.2") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryo.registrator", "org.apache.sedona.core.serde.SedonaKryoRegistrator") \
    .getOrCreate()

# Τώρα φτιάξε το SedonaContext
sedona = SedonaContext.create(spark)

your 131072x1 screen size is bogus. expect trouble
25/12/06 21:33:33 WARN Utils: Your hostname, LAPTOP-POBVNKJ0 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/12/06 21:33:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/fragkiska/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/fragkiska/.ivy2/cache
The jars for the packages stored in: /home/fragkiska/.ivy2/jars
org.apache.sedona#sedona-spark-shaded-3.4_2.12 added as a dependency
org.datasyslab#geotools-wrapper added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f961014a-18e5-4d27-afe5-78dbf4bc1ac0;1.0
	confs: [default]
	found org.apache.sedona#sedona-spark-shaded-3.4_2.12;1.6.1 in central
	found org.datasyslab#geotools-wrapper;1.6.1-28.2 in central
downloading https://repo1.maven.org/maven2/org/apache/sedona/sedona-spark-shaded-3.4_2.12/1.6.1/sedona-spark-shaded-3.4_2.12-1.6.1.jar ...
	[SUCCESSFUL ] org.apache.sedona#sedona-spark-shaded-3.4_2.12;1.6.1!sedona-spark-shaded-3.4_2.12.jar (1929ms)
downloading https://repo1.maven.org/maven2/org/datasyslab/geotools-wrapper/1.6.1-28.2/geotools-wrapper-1.6.1-28.2.jar ...
	[SUCCESSFUL ] org.datasyslab#geotools-wrapper;1.6.1-28.2!geotools-wrapper.jar (2453ms)
:: resolution report :: resolve 2380ms :: a

✓ Τα κατάφερες!


In [ ]:
from pyspark.sql.functions import *


from pathlib import Path

project_root = Path.cwd()

data_dir = project_root / "data"

crime_data_2010_2019 = data_dir / "LA_Crime_Data_2010_2019.csv"
crime_data_2020_2025 = data_dir / "LA_Crime_Data_2020_2025.csv"
stations_path = data_dir / "LA_Police_Stations.csv"

stations = (
    spark.read.csv(str(stations_path), header=True, inferSchema=True)
          .withColumnRenamed("X", "station_lon")
          .withColumnRenamed("Y", "station_lat")
)

df1 = spark.read.csv(str(crime_data_2010_2019), header=True, inferSchema=True)
df2 = spark.read.csv(str(crime_data_2020_2025), header=True, inferSchema=True)

crime_data = df1.unionByName(df2)

crime = crime_data.select("LAT", "LON", "AREA NAME")


In [ ]:
from pyspark.sql.functions import *
from sedona.spark import SedonaContext

# === 1. Crime points ===
crime_geo = (
    crime
    .withColumn(
        "crime_point",
        expr("ST_Point(CAST(LON AS Decimal(24,20)), CAST(LAT AS Decimal(24,20)))")
    )
)

# === 2. Police station points ===
stations_geo = (
    stations
    .withColumn(
        "station_point",
        expr("ST_Point(CAST(station_lon AS Decimal(24,20)), CAST(station_lat AS Decimal(24,20)))")
    )
)

# === 3. Spatial join within 2 km ===
radius_meters = 2000

joined = (
    crime_geo.crossJoin(stations_geo)
    .where(expr(f"ST_DistanceSphere(crime_point, station_point) <= {radius_meters}"))
)

# === 4. Crime count per station ===
crime_counts = (
    joined.groupBy("DIVISION")
          .count()
          .orderBy(desc("count"))
)

crime_counts.show(50, truncate=False)


ERROR:root:KeyboardInterrupt while sending command.                (0 + 8) / 16]
Traceback (most recent call last):
  File "/home/fragkiska/spark-env/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/fragkiska/spark-env/lib/python3.12/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 